# mPES — Entrenamiento en Colab Pro+ (GPU)

Entrena **pes_dqn** (DQN) o **pes_ac** (A2C) en una instancia **GPU** de Colab Pro+.
El proyecto se copia desde Google Drive y los modelos entrenados se respaldan
automáticamente a Drive de forma periódica y al finalizar.

Las dependencias se instalan desde `requirements.txt`, replicando las versiones
exactas de `linux_mpes_env` (Python 3.12).

## Requisitos previos

1. Subir la carpeta `mPES/` a Google Drive.
2. Seleccionar un runtime con **GPU** en Colab (Runtime → Change runtime type → GPU).

## Estructura en Drive

```
MyDrive/
├── mPES/                              ← repo source (código + inputs)
│   ├── requirements.txt
│   ├── pes_dqn/  pes_ac/  ...
└── mPES_results/                      ← outputs organizados
    └── {PACKAGE}/
        └── train/
            └── {TRAIN_DATE}/          ← modelo .keras, rewards, plots
```

## Flujo

1. **Mount Drive** → copia el repo desde `mPES/` a `/content/mPES`
2. **Setup** → instala Python 3.12 + venv con las mismas dependencias que `linux_mpes_env`
3. **Entrenar** → lanza el subproceso con GPU habilitada, sync periódico a Drive
4. **Resultados** → copia los outputs finales a `mPES_results/{PACKAGE}/train/{DATE}/`

### Notas sobre GPU

- La GPU se habilita con `CUDA_VISIBLE_DEVICES='0'`.
- El paquete configura automáticamente *memory growth* para evitar OOM.
- El seed (SEED=42) garantiza reproducibilidad **en CPU**. En GPU puede haber
  variaciones mínimas por operaciones no-deterministas.

In [ ]:
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive')

In [ ]:
#@title **Configuración** { display-mode: "form" }

PACKAGE = "pes_ac" #@param ["pes_dqn", "pes_ac"]
NUM_EPISODES = 50000 #@param {type:"integer"}
DRIVE_REPO = "/content/drive/MyDrive/mPES" #@param {type:"string"}
DRIVE_RESULTS = "/content/drive/MyDrive/mPES_results" #@param {type:"string"}
DRIVE_SYNC_MINUTES = 15 #@param {type:"slider", min:1, max:30, step:1}
NTFY_TOPIC = "mpes-train" #@param {type:"string"}

# --- Derivados ---
import os
from datetime import datetime

_MODULE_MAP = {'pes_dqn': 'train_dqn', 'pes_ac': 'train_ac'}
_SUFFIX_MAP = {'pes_dqn': 'DQN_TRAIN', 'pes_ac': 'AC_TRAIN'}
TRAIN_MODULE = _MODULE_MAP[PACKAGE]
REPO_DIR = '/content/mPES'
TRAIN_DATE = datetime.now().strftime('%Y-%m-%d')
TRAIN_SUBDIR = f'{PACKAGE}/inputs/{TRAIN_DATE}_{_SUFFIX_MAP[PACKAGE]}'

# --- Estructura de resultados en Drive ---
# mPES_results/{PACKAGE}/train/{TRAIN_DATE}/
DRIVE_TRAIN_DIR = os.path.join(DRIVE_RESULTS, PACKAGE, 'train', TRAIN_DATE)

# --- Python 3.12 venv (misma versión que linux_mpes_env) ---
PYTHON_VERSION = '3.12'
VENV_DIR = '/content/mpes_env'
PYTHON_BIN = os.path.join(VENV_DIR, 'bin', 'python')
PIP_BIN = os.path.join(VENV_DIR, 'bin', 'pip')

print(f'Paquete: {PACKAGE}  |  Módulo: {PACKAGE}.ext.{TRAIN_MODULE}')
print(f'Episodios: {NUM_EPISODES:,}  |  Sync: cada {DRIVE_SYNC_MINUTES} min')
print(f'Repo en Drive:      {DRIVE_REPO}')
print(f'Resultados en Drive: {DRIVE_TRAIN_DIR}')
print(f'Python target: {PYTHON_VERSION}  |  Venv: {VENV_DIR}')

## 1. Setup del entorno

Copia el repo desde Drive, instala Python 3.12 en un venv, e instala
todas las dependencias desde `requirements.txt` (mismas versiones que `linux_mpes_env`).

In [ ]:
import os, shutil, subprocess

# --- Copiar repo desde Drive ---
if not os.path.exists(DRIVE_REPO):
    raise FileNotFoundError(
        f'No se encontró el repo en Drive: {DRIVE_REPO}\n'
        f'Subí la carpeta mPES a Google Drive antes de continuar.'
    )

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
shutil.copytree(DRIVE_REPO, REPO_DIR)
print(f'Repo copiado desde Drive ({DRIVE_REPO} → {REPO_DIR})')

# --- Instalar Python 3.12 (misma versión que linux_mpes_env) ---
subprocess.run(
    ['add-apt-repository', '-y', 'ppa:deadsnakes/ppa'],
    check=True, capture_output=True
)
subprocess.run(['apt-get', 'update', '-qq'], check=True, capture_output=True)
subprocess.run([
    'apt-get', 'install', '-y', '-qq',
    f'python{PYTHON_VERSION}', f'python{PYTHON_VERSION}-venv',
    f'python{PYTHON_VERSION}-dev'
], check=True, capture_output=True)
print(f'Python {PYTHON_VERSION} instalado')

# --- Crear venv con Python 3.12 ---
if not os.path.exists(VENV_DIR):
    subprocess.run(
        [f'python{PYTHON_VERSION}', '-m', 'venv', VENV_DIR],
        check=True
    )
    print(f'Venv creado: {VENV_DIR}')
else:
    print(f'Venv existente: {VENV_DIR}')

# --- Instalar dependencias (mismas versiones que linux_mpes_env) ---
req_file = os.path.join(REPO_DIR, 'requirements.txt')
if not os.path.exists(req_file):
    raise FileNotFoundError(f'No se encontró {req_file}')

subprocess.run([PIP_BIN, 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([PIP_BIN, 'install', '-q', '-r', req_file], check=True)
print(f'Dependencias instaladas desde {req_file}')

# --- Verificar versiones y GPU ---
result = subprocess.run([PYTHON_BIN, '-c',
    'import sys, numpy, tensorflow as tf; '
    'gpus = tf.config.list_physical_devices("GPU"); '
    'print(f"Python {sys.version.split()[0]}, numpy {numpy.__version__}, '
    'TF {tf.__version__}"); '
    'print(f"GPU: {gpus[0].name if gpus else \\"ninguna (CPU)\\"}"); '
    '[tf.config.experimental.set_memory_growth(g, True) for g in gpus]; '
    'print(f"Memory growth: configurado") if gpus else None'
], capture_output=True, text=True, check=True)
print(result.stdout.strip())

if 'ninguna' in result.stdout:
    print('\n⚠️  No se detectó GPU. Verifica que el runtime de Colab sea GPU.')
    print('    Runtime → Change runtime type → GPU')
    print('    El entrenamiento funcionará en CPU pero será más lento.')
else:
    print('✅ GPU detectada — el entrenamiento usará aceleración por hardware.')

In [ ]:
import subprocess, os

subprocess.run(['nvidia-smi'], check=False)

with open('/proc/meminfo') as f:
    for line in f:
        if 'MemTotal' in line:
            print(f'\nRAM total: {int(line.split()[1]) / 1024**2:.1f} GB')
            break

# --- Verificar archivos requeridos ---
csv1 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'sequence_lengths.csv')
csv2 = os.path.join(REPO_DIR, PACKAGE, 'inputs', 'initial_severity.csv')

print('\nArchivos requeridos:')
for fpath in [csv1, csv2]:
    ok = os.path.exists(fpath)
    print(f'  {"OK" if ok else "FALTA"}  {os.path.relpath(fpath, REPO_DIR)}')

## 2. Entrenar (GPU)

Lanza el entrenamiento como subproceso usando el venv Python 3.12 con **GPU habilitada**.
Un hilo respalda los outputs en Drive cada `DRIVE_SYNC_MINUTES` minutos.
El script genera el modelo `.keras`, rewards `.npy`, plots y config.
El paquete configura automáticamente *memory growth* en la GPU.

In [ ]:
import subprocess, os, time, threading, shutil
import urllib.request


def ntfy(msg, title=None, priority='default', tags=''):
    """Envia notificacion push via ntfy.sh."""
    if not NTFY_TOPIC:
        return
    try:
        headers = {}
        if title:
            safe = title.encode('latin-1', errors='ignore').decode('latin-1').strip()
            if safe:
                headers['Title'] = safe
        if priority and priority != 'default':
            headers['Priority'] = priority
        if tags:
            headers['Tags'] = tags
        req = urllib.request.Request(
            f'https://ntfy.sh/{NTFY_TOPIC}',
            data=msg.encode('utf-8'), headers=headers, method='POST'
        )
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:
        print(f'  [ntfy] Error: {e}', flush=True)


def fmt_dur(s):
    """Segundos a formato legible."""
    if s < 60: return f'{s:.0f}s'
    if s < 3600: return f'{int(s//60)}m {int(s%60)}s'
    return f'{int(s//3600)}h {int((s%3600)//60)}m'


# --- Hilo de sync periódico a Drive ---
train_dir_src = os.path.join(REPO_DIR, TRAIN_SUBDIR)
done_event = threading.Event()

def sync_to_drive():
    while not done_event.is_set():
        done_event.wait(DRIVE_SYNC_MINUTES * 60)
        if done_event.is_set():
            break
        try:
            if os.path.exists(train_dir_src):
                os.makedirs(DRIVE_TRAIN_DIR, exist_ok=True)
                shutil.copytree(train_dir_src, DRIVE_TRAIN_DIR, dirs_exist_ok=True)
                print(f'  [Drive Sync {time.strftime("%H:%M:%S")}] Outputs respaldados en {DRIVE_TRAIN_DIR}', flush=True)
        except Exception as e:
            print(f'  [Drive Sync] Error: {e}', flush=True)

threading.Thread(target=sync_to_drive, daemon=True).start()

# --- Entorno y comando (usa el venv Python 3.12) ---
env = os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES': '0',
    'VIRTUAL_ENV': VENV_DIR,
    'PATH': os.path.join(VENV_DIR, 'bin') + ':' + env.get('PATH', ''),
    'PYTHONUNBUFFERED': '1',
    'PYTHONIOENCODING': 'utf-8',
    'TF_ENABLE_ONEDNN_OPTS': '0',
    'TF_CPP_MIN_LOG_LEVEL': '2',
})

cmd = [PYTHON_BIN, '-m', f'{PACKAGE}.ext.{TRAIN_MODULE}', str(NUM_EPISODES)]

ntfy(
    f'Entrenamiento iniciado\nPaquete: {PACKAGE}\nEpisodios: {NUM_EPISODES:,}',
    title=f'{PACKAGE} -- Train iniciado', tags='rocket'
)

print(f'▶ {" ".join(cmd)}')
print(f'  cwd: {REPO_DIR}  |  Episodios: {NUM_EPISODES:,}')
print(f'  Drive sync: cada {DRIVE_SYNC_MINUTES} min')
print()

# --- Ejecutar ---
t_start = time.time()

proc = subprocess.Popen(
    cmd, cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

exit_code = proc.wait()
done_event.set()
elapsed = time.time() - t_start

# --- Backup final a Drive ---
if os.path.exists(train_dir_src):
    os.makedirs(DRIVE_TRAIN_DIR, exist_ok=True)
    shutil.copytree(train_dir_src, DRIVE_TRAIN_DIR, dirs_exist_ok=True)

print(f'\n{"="*50}')
print(f'  FINALIZADO  |  Tiempo: {fmt_dur(elapsed)}  |  Exit: {exit_code}')
print(f'{"="*50}')

ntfy(
    f'Entrenamiento finalizado\nPaquete: {PACKAGE}\n'
    f'Episodios: {NUM_EPISODES:,}\nTiempo: {fmt_dur(elapsed)}\nExit: {exit_code}',
    title=f'{PACKAGE} -- Train finalizado',
    priority='high',
    tags='white_check_mark' if exit_code == 0 else 'warning'
)

## 3. Copiar resultados a Drive

Copia los outputs del entrenamiento a la estructura organizada:
`mPES_results/{PACKAGE}/train/{TRAIN_DATE}/`

In [ ]:
import os, shutil

train_dir_src = os.path.join(REPO_DIR, TRAIN_SUBDIR)

if os.path.exists(train_dir_src):
    os.makedirs(DRIVE_TRAIN_DIR, exist_ok=True)
    shutil.copytree(train_dir_src, DRIVE_TRAIN_DIR, dirs_exist_ok=True)
    print(f'Outputs copiados a: {DRIVE_TRAIN_DIR}')
    for fname in sorted(os.listdir(DRIVE_TRAIN_DIR)):
        size = os.path.getsize(os.path.join(DRIVE_TRAIN_DIR, fname))
        print(f'  {fname} ({size/1024:.0f} KB)')
else:
    print(f'No se encontró directorio de salida: {train_dir_src}')
    print('Directorios disponibles:')
    inputs_dir = os.path.join(REPO_DIR, PACKAGE, 'inputs')
    for d in sorted(os.listdir(inputs_dir)):
        if os.path.isdir(os.path.join(inputs_dir, d)):
            print(f'  {d}')